In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from dataset.accident_dataset import get_dataset_paths, resolve_dataset_root

In [ ]:
def parse_cls_results(data, result_name):
    results = []
    for row in data:
        d = {}
        name = result_name.strip('_analysis')
        d[name, 'crop_point', 'prompt1'] = row['crop_point']['cls_prompt1_label']
        d[name, 'crop_point', 'prompt2'] = row['crop_point']['cls_prompt2_label']

        d[name, 'crop_bbox', 'prompt1'] = row['crop_bbox']['cls_prompt1_label']
        d[name, 'crop_bbox', 'prompt2'] = row['crop_bbox']['cls_prompt2_label']

        d[name, 'crop_full', 'prompt1'] = row['crop_full']['cls_prompt1_label']
        d[name, 'crop_full', 'prompt2'] = row['crop_full']['cls_prompt2_label']
        results.append(d)
    return results


def parse_spatial_results(data, result_name):
    results = []

    for row in data:
        x_pred = row['spatial']['x']
        y_pred = row['spatial']['y']

        if (x_pred is None) or (y_pred is None):
            x_pred = row['width'] / 2
            y_pred = row['height'] / 2

        # normalize predictions
        x = x_pred / row['width']
        y = y_pred / row['height']

        # ground truth
        x_gt = row['center_x']
        y_gt = row['center_y']

        # Gaussian components
        x_part = (x - x_gt)**2 / (2 * (SIGMA_X**2))
        y_part = (y - y_gt)**2 / (2 * (SIGMA_Y**2))

        score = np.exp(-(x_part + y_part))
        results.append(score)
    return pd.Series(results, name=result_name)



In [ ]:
# Load GT
dataset_path = Path('../../dataset')
dataset_root = resolve_dataset_root(dataset_path)
_, metadata_path = get_dataset_paths(
    dataset_root, kind="real"
)
df = pd.read_csv(metadata_path)
df["video_path"] = df["path"].apply(lambda x: os.path.join(dataset_root, x))


anns_width = (df["x2"] - df["x1"]).abs()
anns_height = (df["y2"] - df["y1"]).abs()
sigmas = np.stack([anns_width, anns_height,], axis=1)
sigmas = np.mean(sigmas, axis=0)
SIGMA_X, SIGMA_Y = sigmas

# Results dir
root = "./results"
result_names = ['molmo_oracle_analysis', 'molmo_pred_analysis', 'qwen_oracle_analysis', 'qwen_pred_analysis']


## Classification using VLMs

In [ ]:
data = []
for result_name in result_names:
    outputs = torch.load(f"{root}/{result_name}.pkl", weights_only=False)    
    data.append(pd.DataFrame(parse_cls_results(outputs, result_name)))
df_results = pd.concat(data, axis=1)
df_results = df_results.fillna('none')

gt = df['type'].values
df_means = (df_results == gt[:, None]).mean()

In [ ]:
# Table 4 results
display(df_means.loc[[
    ('qwen_oracle', 'crop_bbox', 'prompt2'),
    ('qwen_oracle', 'crop_full', 'prompt2'),
    ('molmo_oracle', 'crop_bbox', 'prompt2'),
    ('molmo_oracle', 'crop_full', 'prompt2'),
]])

In [ ]:
# Table 5 results (classification results for Molmo)
display(df_means.loc[[
    ('molmo_oracle', 'crop_full', 'prompt2'),
    ('molmo_pred', 'crop_full', 'prompt2'),
]])

## Spatial localization using VLMs

In [ ]:
data = []
for result_name in result_names:
    outputs = torch.load(f"{root}/{result_name}.pkl", weights_only=False)
    data.append(parse_spatial_results(outputs, result_name))
df_results = pd.concat(data, axis=1)

df_results.mean()

In [ ]:
data = []
for result_name in result_names:
    outputs = torch.load(f"{root}/{result_name}.pkl", weights_only=False)
    data.append(parse_spatial_results(outputs, result_name))
df_results = pd.concat(data, axis=1)

df_results.mean()